<a href="https://colab.research.google.com/github/Shemuel-bot/AI-ball-notebooks/blob/main/Ball_Trap_Trainability_Analyzer_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ball Trap Brain — Exact Trainability Analyzer (v2)

This notebook is built against the **uploaded `Ball_Trap_Brain(1).ipynb` implementation**, including its exact:

- `TokenizeInput()`
- leave-one-out, position-weighted `travel()`
- `compute_input()`
- `ricochet()`
- `ricochet_deflectors()`

The purpose is to separate three different questions:

1. **Architecture/data impossibility:** can we prove that no parameter assignment can fit the dataset?
2. **Exact feasibility:** within a declared finite parameter domain, does a satisfying brain exist?
3. **Optimizer performance:** if a solution exists, did a particular training algorithm actually find it?

The analyzer never treats "my optimizer failed" as proof that the architecture is untrainable.

For large brains, exhaustive Cartesian search is generally impossible. The notebook therefore adds an **optional Z3/SMT backend** that symbolically unrolls the bounded ricochet computation. If Z3 is unavailable, the structural analyzer and small-domain exact enumerator still work.


In [ ]:
# Core imports
from __future__ import annotations

from dataclasses import dataclass, field
from itertools import product
from collections import defaultdict, Counter
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple
from math import log10
import random
import copy
import json
import math

try:
    import z3
    Z3_AVAILABLE = True
except ImportError:
    z3 = None
    Z3_AVAILABLE = False

print("Z3 available:", Z3_AVAILABLE)
print("Core analyzer loaded.")


Z3 available: False
Core analyzer loaded.


## 1. Exact copy of your tokenizer and brain computation

In [ ]:
# ---------------------------------------------------------------------------
# Exact tokenizer from the uploaded Ball_Trap_Brain(1).ipynb
# ---------------------------------------------------------------------------

def TokenizeInput(input, alphabet):
    input_str = str(input)
    matches = []

    for order, item in enumerate(alphabet):
        item_str = str(item)
        if not item_str:
            continue

        start_pos = 0
        while True:
            index = input_str.find(item_str, start_pos)
            if index == -1:
                break

            matches.append((index, -len(item_str), order, item))
            start_pos = index + 1

    matches.sort(key=lambda match: (match[0], match[1], match[2]))

    non_overlapping_items = []
    last_end_index = 0

    for match in matches:
        start_index = match[0]
        length = -match[1]
        item = match[3]

        if start_index >= last_end_index:
            non_overlapping_items.append(item)
            last_end_index = start_index + length

    return non_overlapping_items


# ---------------------------------------------------------------------------
# Exact reference copy of the uploaded Ball_Trap_Brain class.
# This intentionally preserves its semantics, including:
#   * weighted leave-one-out travel + Python round()
#   * val being a "hole" only when it is NOT an int
#   * cap-limited ricochet
# ---------------------------------------------------------------------------

class Ball_Trap_Brain:
    def __init__(self, deflectors, bounce, alphabet, output, holes_multiplier=1, direction_range=10):
        self.output = output
        self.holes_multiplier = holes_multiplier
        self.ball_energy = bounce*2
        self.alphabet = alphabet
        self.direction_range = direction_range
        self.alphabet_directions = self.choose_alphabet_direction(len(alphabet))
        self.alphabet_start_positions = self.start_positions(len(alphabet), deflectors)
        self.deflectors_list = [[random.randint(0, deflectors-1), bounce] for i in range(deflectors)]
        self.set_holes(output, deflectors, self.holes_multiplier)

    def set_holes(self, output, deflectors, holes_multiplier):
        for i in range(len(output)):
            for j in range(holes_multiplier):
                self.deflectors_list[random.randint(0, deflectors-1)][1] = output[i]

    def choose_alphabet_direction(self, alphabet):
        return [random.randint(-self.direction_range, self.direction_range) for _ in range(alphabet)]

    def start_positions(self, alphabet, deflectors):
        return [random.randint(0, deflectors-1) for i in range(alphabet)]

    def compute_input(self, input):
        directions = []
        positions = []
        travel = []
        input_arr = TokenizeInput(input, self.alphabet)

        for i in input_arr:
            if i in self.alphabet:
                index = self.alphabet.index(i)
                directions.append(self.alphabet_directions[index])
                positions.append(self.alphabet_start_positions[index])

        for i in range(len(directions)):
            travel.append(self.travel(copy.copy(directions), positions[i], i))

        return self.ricochet(travel, 10, self.ball_energy)

    def ricochet(self, travel, cap, ball_energy):
        output = []
        dl = self.deflectors_list
        n = len(dl)

        for t in travel:
            new_idx = t % n
            cap_idx = 0
            be = ball_energy

            while be > 0 and cap_idx < cap:
                dest, val = dl[new_idx]
                distance = abs(dest - new_idx)

                if not isinstance(val, int):
                    output.append(val)
                    be += 1
                    be -= distance
                    new_idx = dest
                    cap_idx += 1
                    continue

                be += val
                be -= distance
                new_idx = dest
                cap_idx += 1

        return output

    def compute_deflectors(self, input):
        directions = []
        positions = []
        travel = []
        input_arr = TokenizeInput(input, self.alphabet)

        for i in input_arr:
            if i in self.alphabet:
                index = self.alphabet.index(i)
                directions.append(self.alphabet_directions[index])
                positions.append(self.alphabet_start_positions[index])

        for i in range(len(directions)):
            travel.append(self.travel(copy.copy(directions), positions[i], i))

        return self.ricochet_deflectors(travel, 10, self.ball_energy)

    def ricochet_deflectors(self, travel, cap, ball_energy):
        deflectors_hit = []
        dl = self.deflectors_list
        n = len(dl)

        for t in travel:
            new_idx = t % n
            cap_idx = 0
            be = ball_energy

            while be > 0 and cap_idx < cap:
                dest, val = dl[new_idx]
                deflectors_hit.append(new_idx)
                distance = abs(dest - new_idx)

                if not isinstance(val, int):
                    be += 1
                    be -= distance
                    new_idx = dest
                    cap_idx += 1
                    continue

                be += val
                be -= distance
                new_idx = dest
                cap_idx += 1

        return deflectors_hit

    def travel(self, directions, position, index):
        total = 0
        weight_sum = 0

        for j, d in enumerate(directions):
            if j == index:
                continue

            weight = j + 1
            total += d * weight
            weight_sum += weight

        if weight_sum > 0:
            total = round(total / weight_sum)

        position += total
        return position


In [ ]:
# Optional compatibility check against the exact uploaded implementation.
# This should be run only if the notebook is executing from a copy whose class
# was imported separately. The cell below performs deterministic self-checks.

random.seed(1234)
b = Ball_Trap_Brain(
    deflectors=100,
    bounce=2,
    alphabet=("A", "B", "C"),
    output=("X", "Y"),
    holes_multiplier=1,
    direction_range=3,
)



print("Tokens:", TokenizeInput("ABAC", b.alphabet))
print("travel([2,-1,3], 3, 1) =", b.travel([2, -1, 3], 3, 1))
print("compute_input('ABC') =", b.compute_input("ABC"))
print("compute_deflectors('ABC') =", b.compute_deflectors("ABC"))



Tokens: ['A', 'B', 'A', 'C']
travel([2,-1,3], 3, 1) = 6
compute_input('ABC') = []
compute_deflectors('ABC') = [98, 9, 75]


## 2. Mathematical model used by the analyzer

For a finite dataset

\[
D=\{(x_1,y_1),\ldots,(x_m,y_m)\}
\]

and a declared finite parameter domain \(\Theta\), exact feasibility is

\[
\exists\theta\in\Theta\quad
\forall k,\; f_\theta(x_k)=y_k.
\]

The result is interpreted as:

- **EXACT_FEASIBLE:** a satisfying assignment was found.
- **EXACT_INFEASIBLE:** the entire declared finite domain was exhausted, or a parameter-independent contradiction proves impossibility.
- **UNRESOLVED:** the domain was not fully searched.

This distinction is essential. A failed evolutionary run is not an infeasibility proof.


In [ ]:
@dataclass(frozen=True)
class Example:
    input_value: Any
    target: Tuple[Any, ...]

    @classmethod
    def from_pair(cls, x, y):
        if isinstance(y, (list, tuple)):
            target = tuple(y)
        else:
            target = (y,)
        return cls(x, target)


def make_examples(pairs):
    return [Example.from_pair(x, y) for x, y in pairs]


@dataclass(frozen=True)
class BrainParams:
    alphabet_directions: Tuple[int, ...]
    alphabet_start_positions: Tuple[int, ...]
    destinations: Tuple[int, ...]
    values: Tuple[Any, ...]


@dataclass
class AnalysisReport:
    status: str
    reason: str = ""
    assignments_checked: int = 0
    total_assignments: Optional[int] = None
    witness: Optional[BrainParams] = None
    conflicting_example_ids: List[int] = field(default_factory=list)
    structural_findings: List[str] = field(default_factory=list)
    best_accuracy: Optional[float] = None
    backend: str = ""


def _value_is_hole(v):
    # EXACT uploaded semantics: only non-int values emit output.
    return not isinstance(v, int)


class ExactBallTrapModel:
    """Deterministic model matching the uploaded Ball_Trap_Brain class."""

    def __init__(self, alphabet, deflector_count, bounce=2, cap=10, direction_range=10):
        self.alphabet = tuple(alphabet)
        self.n = int(deflector_count)
        self.bounce = int(bounce)
        self.cap = int(cap)
        self.direction_range = int(direction_range)

        if self.n <= 0:
            raise ValueError("deflector_count must be positive")
        if self.cap < 0:
            raise ValueError("cap must be nonnegative")

    def tokenize(self, x):
        return TokenizeInput(x, self.alphabet)

    def travel(self, directions, position, index):
        total = 0
        weight_sum = 0
        for j, d in enumerate(directions):
            if j == index:
                continue
            weight = j + 1
            total += d * weight
            weight_sum += weight
        if weight_sum > 0:
            total = round(total / weight_sum)
        position += total
        return position

    def compute(self, x, p: BrainParams, cap=None, return_trace=False):
        cap = self.cap if cap is None else int(cap)

        input_arr = self.tokenize(x)
        alphabet_index = {item: i for i, item in enumerate(self.alphabet)}

        directions = [
            p.alphabet_directions[alphabet_index[token]]
            for token in input_arr
            if token in alphabet_index
        ]
        positions = [
            p.alphabet_start_positions[alphabet_index[token]]
            for token in input_arr
            if token in alphabet_index
        ]

        travels = [
            self.travel(list(directions), positions[i], i)
            for i in range(len(directions))
        ]

        output = []
        trace = []

        for launch, t in enumerate(travels):
            new_idx = t % self.n
            be = self.bounce * 2
            steps = 0
            launch_trace = []

            while be > 0 and steps < cap:
                dest = p.destinations[new_idx]
                val = p.values[new_idx]
                distance = abs(dest - new_idx)

                emitted = None
                if _value_is_hole(val):
                    emitted = val
                    output.append(val)
                    be += 1
                    be -= distance
                else:
                    be += val
                    be -= distance

                launch_trace.append({
                    "step": steps,
                    "index": new_idx,
                    "dest": dest,
                    "value": val,
                    "distance": distance,
                    "emitted": emitted,
                    "energy_after": be,
                })

                new_idx = dest
                steps += 1

            trace.append({
                "launch": launch,
                "travel": t,
                "start_index": t % self.n,
                "steps": launch_trace,
                "output_after_launch": tuple(output),
            })

        return (tuple(output), trace) if return_trace else tuple(output)

    def parameter_space_size(self, direction_min=None, direction_max=None,
                             value_domain=None):
        lo = self.direction_range * -1 if direction_min is None else int(direction_min)
        hi = self.direction_range if direction_max is None else int(direction_max)
        directions = hi - lo + 1
        starts = self.n
        destinations = self.n

        if value_domain is None:
            raise ValueError("Provide value_domain = (bounce plus output values) for exact size.")
        values = len(tuple(value_domain))

        return directions ** len(self.alphabet) * starts ** len(self.alphabet) * \
               destinations ** self.n * values ** self.n

    def parameter_space_log10(self, direction_min=None, direction_max=None,
                              value_domain=None):
        size = self.parameter_space_size(direction_min, direction_max, value_domain)
        return log10(size) if size > 0 else float("-inf")


In [ ]:
class TrainabilityAnalyzer:
    def __init__(self, model: ExactBallTrapModel, examples: Sequence[Example],
                 direction_min=None, direction_max=None, value_domain=None,
                 exact_max_assignments=2_000_000):
        self.model = model
        self.examples = list(examples)

        self.direction_min = -model.direction_range if direction_min is None else int(direction_min)
        self.direction_max = model.direction_range if direction_max is None else int(direction_max)

        if self.direction_min > self.direction_max:
            raise ValueError("direction_min > direction_max")

        # The uploaded constructor begins with bounce at every deflector and
        # set_holes can replace values with entries from output. Mutation also
        # chooses from output. Therefore this is the true behavioral domain.
        if value_domain is None:
            value_domain = (model.bounce,)
        self.value_domain = tuple(dict.fromkeys(value_domain))

        self.exact_max_assignments = int(exact_max_assignments)

        if not self.value_domain:
            raise ValueError("value_domain cannot be empty")

    # ---------------------------
    # Structural / proof checks
    # ---------------------------

    def structural_checks(self):
        findings = []

        # Tokenization equivalence classes.
        by_tokens = defaultdict(list)
        for i, ex in enumerate(self.examples):
            by_tokens[tuple(self.model.tokenize(ex.input_value))].append(i)

        for toks, ids in by_tokens.items():
            targets = {self.examples[i].target for i in ids}
            if len(targets) > 1:
                findings.append(
                    f"IMPOSSIBLE: examples {ids} have identical tokenization {toks!r} "
                    f"but require different targets."
                )

        reachable_emitted_values = tuple(v for v in self.value_domain if _value_is_hole(v))

        for i, ex in enumerate(self.examples):
            token_count = len(self.model.tokenize(ex.input_value))
            target_len = len(ex.target)

            if target_len > token_count * self.model.cap:
                findings.append(
                    f"IMPOSSIBLE: example {i} has {target_len} target outputs but only "
                    f"{token_count} token-launches × cap {self.model.cap} = "
                    f"{token_count * self.model.cap} emissions are possible."
                )

            if token_count == 0 and target_len > 0:
                findings.append(
                    f"IMPOSSIBLE: example {i} contains zero recognized tokens but "
                    f"requires {target_len} outputs."
                )

            illegal = [v for v in ex.target if v not in reachable_emitted_values]
            if illegal:
                findings.append(
                    f"IMPOSSIBLE: example {i} requests output values {illegal!r} that "
                    f"cannot be emitted under the declared value domain. "
                    f"Emittable values are {reachable_emitted_values!r}."
                )

        # Identical target groups are useful but not contradictions.
        repeated = {k: v for k, v in by_tokens.items() if len(v) > 1}
        for toks, ids in repeated.items():
            if len({self.examples[i].target for i in ids}) == 1:
                findings.append(
                    f"INFO: examples {ids} collapse to the same tokenization {toks!r}; "
                    "this is expected to make them functionally identical."
                )

        return findings

    def provably_infeasible(self):
        findings = self.structural_checks()
        impossible = [x for x in findings if x.startswith("IMPOSSIBLE:")]
        return impossible, findings

    # ---------------------------
    # Full finite enumeration
    # ---------------------------

    def _candidate_parameters(self):
        directions = list(product(
            range(self.direction_min, self.direction_max + 1),
            repeat=len(self.model.alphabet)
        ))
        starts = list(product(range(self.model.n), repeat=len(self.model.alphabet)))
        destinations = list(product(range(self.model.n), repeat=self.model.n))
        values = list(product(self.value_domain, repeat=self.model.n))

        for d, s, dest, val in product(directions, starts, destinations, values):
            yield BrainParams(tuple(d), tuple(s), tuple(dest), tuple(val))

    def evaluate(self, p):
        correct = 0
        predictions = []
        for ex in self.examples:
            pred = self.model.compute(ex.input_value, p)
            predictions.append(pred)
            if pred == ex.target:
                correct += 1
        accuracy = correct / len(self.examples) if self.examples else 1.0
        return accuracy, predictions

    def exact_enumeration(self, stop_at_first=True):
        impossible, findings = self.provably_infeasible()
        if impossible:
            return AnalysisReport(
                status="EXACT_INFEASIBLE",
                reason="A parameter-independent contradiction was found.",
                total_assignments=0,
                structural_findings=findings,
                backend="structural-proof",
            )

        total = self.model.parameter_space_size(
            self.direction_min, self.direction_max, self.value_domain
        )
        if total > self.exact_max_assignments:
            return AnalysisReport(
                status="UNRESOLVED",
                reason=f"Exact Cartesian search has {total:,} assignments, above the "
                       f"configured limit of {self.exact_max_assignments:,}.",
                total_assignments=total,
                structural_findings=findings,
                backend="enumeration",
            )

        checked = 0
        best_acc = -1.0
        best = None

        for p in self._candidate_parameters():
            checked += 1
            acc, _ = self.evaluate(p)

            if acc > best_acc:
                best_acc, best = acc, p

            if acc == 1.0:
                return AnalysisReport(
                    status="EXACT_FEASIBLE",
                    reason="A satisfying parameter assignment was found.",
                    assignments_checked=checked,
                    total_assignments=total,
                    witness=p,
                    structural_findings=findings,
                    best_accuracy=1.0,
                    backend="enumeration",
                )

        return AnalysisReport(
            status="EXACT_INFEASIBLE",
            reason="Every assignment in the declared finite parameter domain was checked.",
            assignments_checked=checked,
            total_assignments=total,
            structural_findings=findings,
            best_accuracy=best_acc,
            backend="enumeration",
        )

    # ---------------------------
    # Witness inspection
    # ---------------------------

    def inspect(self, witness: BrainParams):
        rows = []
        for i, ex in enumerate(self.examples):
            pred, trace = self.model.compute(ex.input_value, witness, return_trace=True)
            rows.append({
                "example_id": i,
                "input": ex.input_value,
                "tokens": self.model.tokenize(ex.input_value),
                "target": ex.target,
                "prediction": pred,
                "correct": pred == ex.target,
                "trace": trace,
            })
        return rows

    @staticmethod
    def summarize_witness(w: BrainParams):
        return {
            "alphabet_directions": w.alphabet_directions,
            "alphabet_start_positions": w.alphabet_start_positions,
            "destinations": w.destinations,
            "values": w.values,
        }

    def report(self, r: AnalysisReport):
        lines = [
            f"STATUS: {r.status}",
            f"BACKEND: {r.backend}",
            f"REASON: {r.reason}",
        ]
        if r.total_assignments is not None:
            lines.append(f"PARAMETER SPACE: {r.total_assignments:,} assignments")
        if r.assignments_checked:
            lines.append(f"ASSIGNMENTS CHECKED: {r.assignments_checked:,}")
        if r.best_accuracy is not None:
            lines.append(f"BEST ACCURACY FOUND: {r.best_accuracy:.4f}")
        if r.structural_findings:
            lines.append("\nFINDINGS:")
            lines.extend("  " + f for f in r.structural_findings)
        if r.witness is not None:
            lines.append("\nWITNESS:")
            lines.append(json.dumps(self.summarize_witness(r.witness), indent=2, default=str))
        return "\n".join(lines)


## 3. Optional exact SMT backend

When Z3 is installed, the analyzer can encode the bounded computation symbolically.

The encoding represents:

- every alphabet direction as an integer variable;
- every alphabet start position as an integer variable;
- every deflector destination as an integer variable;
- every deflector value as a finite-domain choice;
- the exact `travel()` weighted average and Python round semantics;
- the modulo starting index;
- every bounded ricochet step;
- emitted-output sequence and output length.

This is still a **bounded exact model**: the `cap` is finite and the parameter domains are explicitly bounded.

A satisfying model is a mathematical witness. `UNSAT` is an actual proof that no assignment inside the declared domain can satisfy the constraints.

If Z3 is not installed in your environment, install `z3-solver` once and rerun the import cell.


In [ ]:
def _round_constraints_z3(x, denominator, k):
    """Return exact conditions for Python's round(x / denominator) == k.

    x and denominator are integers, denominator > 0.
    Python uses ties-to-even.
    """
    # x/denominator lies between k-0.5 and k+0.5.
    lower_num = (2*k - 1) * denominator
    upper_num = (2*k + 1) * denominator

    if k % 2 == 0:
        lower = 2*x >= lower_num
        upper = 2*x <= upper_num
    else:
        lower = 2*x > lower_num
        upper = 2*x < upper_num

    return z3.And(lower, upper)


class Z3TrainabilityAnalyzer(TrainabilityAnalyzer):
    def _require_z3(self):
        if not Z3_AVAILABLE:
            raise RuntimeError(
                "Z3 is not installed. Install `z3-solver`, restart the kernel, "
                "and rerun the import cell."
            )

    def _build_solver(self, track_examples=True):
        self._require_z3()
        s = z3.Solver()

        # Alphabet parameters.
        dirs = [
            z3.Int(f"dir_{i}") for i in range(len(self.model.alphabet))
        ]
        starts = [
            z3.Int(f"start_{i}") for i in range(len(self.model.alphabet))
        ]

        for v in dirs:
            s.add(v >= self.direction_min, v <= self.direction_max)

        for v in starts:
            s.add(v >= 0, v < self.model.n)

        # Deflector parameters.
        dests = [z3.Int(f"dest_{i}") for i in range(self.model.n)]
        choices = [z3.Int(f"choice_{i}") for i in range(self.model.n)]

        for v in dests:
            s.add(v >= 0, v < self.model.n)
        for v in choices:
            s.add(v >= 0, v < len(self.value_domain))

        # Finite-domain decode functions.
        def hole_expr(choice):
            clauses = []
            for k, val in enumerate(self.value_domain):
                clauses.append(z3.And(choice == k, _value_is_hole(val)))
            return z3.Or(*clauses) if clauses else z3.BoolVal(False)

        def delta_expr(choice):
            # Exact uploaded semantics:
            # non-int value -> +1 energy
            # int value -> +that integer
            expr = z3.IntVal(0)
            for k, val in enumerate(self.value_domain):
                delta = 1 if _value_is_hole(val) else int(val)
                expr = z3.If(choice == k, z3.IntVal(delta), expr)
            return expr

        def output_code_expr(choice):
            # Code only for values that actually emit.
            output_values = [v for v in self.value_domain if _value_is_hole(v)]
            code = {v: i for i, v in enumerate(output_values)}
            expr = z3.IntVal(-1)
            for k, val in enumerate(self.value_domain):
                if _value_is_hole(val):
                    expr = z3.If(choice == k, z3.IntVal(code[val]), expr)
            return expr

        # Target output codebook.
        emitted_values = [v for v in self.value_domain if _value_is_hole(v)]
        emitted_code = {v: i for i, v in enumerate(emitted_values)}

        alphabet_index = {item: i for i, item in enumerate(self.model.alphabet)}

        # We chain a symbolic global output buffer across all examples.
        max_total_slots = sum(
            len(self.model.tokenize(ex.input_value)) * self.model.cap
            for ex in self.examples
        )

        out0 = z3.K(z3.IntSort(), z3.IntVal(-1))
        len0 = z3.IntVal(0)

        current_out = out0
        current_len = len0

        assumptions = []

        for ex_i, ex in enumerate(self.examples):
            tokens = self.model.tokenize(ex.input_value)
            idxs = [alphabet_index[t] for t in tokens]
            directions = [dirs[i] for i in idxs]
            positions = [starts[i] for i in idxs]

            # First compute each launch's exact travel.
            travels = []
            for launch_i in range(len(tokens)):
                weighted_sum = z3.IntVal(0)
                weight_sum = 0
                for j, d in enumerate(directions):
                    if j == launch_i:
                        continue
                    w = j + 1
                    weighted_sum = weighted_sum + w*d
                    weight_sum += w

                if weight_sum == 0:
                    offset = z3.IntVal(0)
                else:
                    # Bounded direction range gives a finite offset range.
                    kvars = []
                    for k in range(self.direction_min, self.direction_max + 1):
                        kvars.append(
                            z3.If(
                                _round_constraints_z3(weighted_sum, weight_sum, k),
                                z3.IntVal(k),
                                z3.IntVal(0)
                            )
                        )
                    # Exactly one k is valid for integer weighted_sum.
                    offset = z3.Sum(kvars)

                    # Guard against accidental zero-sum ambiguity by requiring
                    # at least one rounding region to hold.
                    s.add(z3.Or(*[
                        _round_constraints_z3(weighted_sum, weight_sum, k)
                        for k in range(self.direction_min, self.direction_max + 1)
                    ]))

                travels.append(positions[launch_i] + offset)

            # Simulate each launch for exactly cap symbolic steps, freezing state
            # after energy reaches zero.
            for launch_i, t in enumerate(travels):
                idx = z3.Mod(t, self.model.n)
                be = z3.IntVal(self.model.bounce * 2)

                for step in range(self.model.cap):
                    active = be > 0

                    dest = z3.Sum([
                        z3.If(idx == j, dests[j], z3.IntVal(0))
                        for j in range(self.model.n)
                    ])

                    choice = z3.Sum([
                        z3.If(idx == j, choices[j], z3.IntVal(0))
                        for j in range(self.model.n)
                    ])

                    distance = z3.Abs(dest - idx)
                    is_hole = hole_expr(choice)
                    delta = delta_expr(choice)
                    out_code = output_code_expr(choice)

                    next_be = z3.If(
                        active,
                        be + delta - distance,
                        be
                    )
                    next_idx = z3.If(active, dest, idx)

                    # Append output exactly when active AND the selected value
                    # is a non-int hole.
                    append = z3.And(active, is_hole)

                    new_len = z3.If(append, current_len + 1, current_len)
                    new_out = z3.If(
                        append,
                        z3.Store(current_out, current_len, out_code),
                        current_out,
                    )

                    # We only care about the first max_total_slots positions.
                    current_out = new_out
                    current_len = new_len
                    be = next_be
                    idx = next_idx

            # At the end of each example, its output must equal its target.
            expected = ex.target

            conditions = [current_len == len(expected)]

            for j, target_value in enumerate(expected):
                target_code = emitted_code[target_value]
                conditions.append(z3.Select(current_out, j) == target_code)

            if track_examples:
                tag = z3.Bool(f"example_{ex_i}")
                assumptions.append(tag)
                s.assert_and_track(z3.And(*conditions), tag)
            else:
                s.add(z3.And(*conditions))

        return s, assumptions, dirs, starts, dests, choices, emitted_values

    def exact_smt(self, get_unsat_core=True):
        self._require_z3()

        impossible, findings = self.provably_infeasible()
        if impossible:
            return AnalysisReport(
                status="EXACT_INFEASIBLE",
                reason="A parameter-independent contradiction was found.",
                total_assignments=0,
                structural_findings=findings,
                backend="structural-proof",
            )

        s, assumptions, dirs, starts, dests, choices, emitted_values = self._build_solver(
            track_examples=True
        )

        result = s.check(*assumptions)

        if result == z3.sat:
            m = s.model()
            witness = BrainParams(
                alphabet_directions=tuple(int(m[v].as_long()) for v in dirs),
                alphabet_start_positions=tuple(int(m[v].as_long()) for v in starts),
                destinations=tuple(int(m[v].as_long()) for v in dests),
                values=tuple(self.value_domain[int(m[v].as_long())] for v in choices),
            )

            # Independent numerical verification against the exact Python model.
            acc, _ = self.evaluate(witness)
            assert acc == 1.0, (
                "SMT witness failed independent Python verification. "
                "This indicates a bug in the SMT encoding."
            )

            return AnalysisReport(
                status="EXACT_FEASIBLE",
                reason="Z3 found a satisfying assignment; the witness independently "
                       "reproduces every target in the exact Python model.",
                witness=witness,
                structural_findings=findings,
                best_accuracy=1.0,
                backend="z3-smt",
            )

        if result == z3.unsat:
            core_ids = []
            if get_unsat_core:
                core = set(str(x) for x in s.unsat_core())
                for i, tag in enumerate(assumptions):
                    if str(tag) in core:
                        core_ids.append(i)

            return AnalysisReport(
                status="EXACT_INFEASIBLE",
                reason="Z3 proved the declared bounded model UNSAT.",
                conflicting_example_ids=core_ids,
                structural_findings=findings,
                backend="z3-smt",
            )

        return AnalysisReport(
            status="UNRESOLVED",
            reason="The SMT solver returned UNKNOWN.",
            structural_findings=findings,
            backend="z3-smt",
        )


## 4. A first sanity test: an exactly solvable toy brain

In [ ]:
# Construct a tiny known brain.
toy_model = ExactBallTrapModel(
    alphabet=("A", "B"),
    deflector_count=2,
    bounce=1,
    cap=2,
    direction_range=1,
)

known = BrainParams(
    alphabet_directions=(0, 1),
    alphabet_start_positions=(0, 1),
    destinations=(1, 0),
    values=("X", 1),
)

toy_pairs = [
    ("AA", toy_model.compute("AA", known)),
    ("AB", toy_model.compute("AB", known)),
    ("BA", toy_model.compute("BA", known)),
]

toy_examples = make_examples(toy_pairs)

toy_analyzer = TrainabilityAnalyzer(
    toy_model,
    toy_examples,
    direction_min=-1,
    direction_max=1,
    value_domain=(1, "X"),
    exact_max_assignments=2_000_000,
)

print("Known target pairs:")
for e in toy_examples:
    print(" ", e.input_value, "->", e.target)

toy_report = toy_analyzer.exact_enumeration()
print("\n" + toy_analyzer.report(toy_report))


Known target pairs:
  AA -> ('X', 'X')
  AB -> ('X', 'X')
  BA -> ('X', 'X')

STATUS: EXACT_FEASIBLE
BACKEND: enumeration
REASON: A satisfying parameter assignment was found.
PARAMETER SPACE: 576 assignments
ASSIGNMENTS CHECKED: 2
BEST ACCURACY FOUND: 1.0000

WITNESS:
{
  "alphabet_directions": [
    -1,
    -1
  ],
  "alphabet_start_positions": [
    0,
    0
  ],
  "destinations": [
    0,
    0
  ],
  "values": [
    1,
    "X"
  ]
}


## 5. A guaranteed contradiction from tokenization

In [ ]:
# "AB!" and "AB?" both tokenize as ("A","B").
# A deterministic brain cannot map the same internal computation to two different outputs.

collision_examples = make_examples([
    ("AB!", ("X",)),
    ("AB?", ("Y",)),
])

collision_analyzer = TrainabilityAnalyzer(
    toy_model,
    collision_examples,
    direction_min=-1,
    direction_max=1,
    value_domain=(1, "X", "Y"),
)

collision_report = collision_analyzer.exact_enumeration()
print(collision_analyzer.report(collision_report))


STATUS: EXACT_INFEASIBLE
BACKEND: structural-proof
REASON: A parameter-independent contradiction was found.
PARAMETER SPACE: 0 assignments

FINDINGS:
  IMPOSSIBLE: examples [0, 1] have identical tokenization ('A', 'B') but require different targets.


## 6. Output-length and output-domain proofs

In [ ]:
length_examples = make_examples([
    ("A", ("X", "X", "X", "X", "X")),
])

length_analyzer = TrainabilityAnalyzer(
    ExactBallTrapModel(
        alphabet=("A",),
        deflector_count=2,
        bounce=1,
        cap=2,
        direction_range=1,
    ),
    length_examples,
    direction_min=-1,
    direction_max=1,
    value_domain=(1, "X"),
)

print(length_analyzer.report(length_analyzer.exact_enumeration()))

print("\n--- Integer output values are NOT holes in your current code ---")
integer_target = make_examples([("A", (7,))])

integer_analyzer = TrainabilityAnalyzer(
    ExactBallTrapModel(
        alphabet=("A",),
        deflector_count=2,
        bounce=1,
        cap=2,
        direction_range=1,
    ),
    integer_target,
    direction_min=-1,
    direction_max=1,
    value_domain=(1, 7),
)

print(integer_analyzer.report(integer_analyzer.exact_enumeration()))


STATUS: EXACT_INFEASIBLE
BACKEND: structural-proof
REASON: A parameter-independent contradiction was found.
PARAMETER SPACE: 0 assignments

FINDINGS:
  IMPOSSIBLE: example 0 has 5 target outputs but only 1 token-launches × cap 2 = 2 emissions are possible.

--- Integer output values are NOT holes in your current code ---
STATUS: EXACT_INFEASIBLE
BACKEND: structural-proof
REASON: A parameter-independent contradiction was found.
PARAMETER SPACE: 0 assignments

FINDINGS:
  IMPOSSIBLE: example 0 requests output values [7] that cannot be emitted under the declared value domain. Emittable values are ().


## 7. Exact SMT test (when Z3 is installed)

In [ ]:
if Z3_AVAILABLE:
    z3_analyzer = Z3TrainabilityAnalyzer(
        toy_model,
        toy_examples,
        direction_min=-1,
        direction_max=1,
        value_domain=(1, "X"),
    )
    z3_report = z3_analyzer.exact_smt()
    print(z3_analyzer.report(z3_report))

    if z3_report.witness:
        print("\nIndependent witness verification:")
        for row in z3_analyzer.inspect(z3_report.witness):
            print(row["input"], "target=", row["target"],
                  "prediction=", row["prediction"],
                  "correct=", row["correct"])
else:
    print(
        "Z3 is not installed in this runtime. "
        "Install `z3-solver` in your local notebook environment to enable "
        "the exact symbolic backend."
    )


Z3 is not installed in this runtime. Install `z3-solver` in your local notebook environment to enable the exact symbolic backend.


## 8. Unsat-core analysis: find a small conflicting subset

With Z3, the analyzer tracks each training example separately.

When the whole dataset is impossible, `unsat_core()` can identify a subset of examples that is already contradictory.

This is useful because it turns:

> "the brain cannot learn this dataset"

into something closer to:

> "these particular examples cannot be jointly represented under the current architecture and parameter domain."

The returned core is a proof-relevant diagnostic, not a claim that every example in the full dataset is individually problematic.


In [ ]:
if Z3_AVAILABLE:
    core_examples = make_examples([
        ("AB!", ("X",)),
        ("AB?", ("Y",)),
        ("BA", ("X",)),
    ])

    core_analyzer = Z3TrainabilityAnalyzer(
        toy_model,
        core_examples,
        direction_min=-1,
        direction_max=1,
        value_domain=(1, "X", "Y"),
    )
    core_report = core_analyzer.exact_smt()
    print(core_analyzer.report(core_report))
    print("Conflict/core example IDs:", core_report.conflicting_example_ids)
else:
    print("Install z3-solver to run unsat-core analysis.")


Install z3-solver to run unsat-core analysis.


## 9. Plug in the real Ball Trap Brain dataset

Use **exactly the parameter domains you actually allow during training**.

For the current uploaded class:

- `direction` is an integer in `[-direction_range, direction_range]`;
- `start_position` is any deflector index;
- `destination` is any deflector index;
- deflector values begin as `bounce` and can be replaced with entries from `output`.

Thus the behavioral value domain for exact analysis is:

```python
value_domain = tuple(dict.fromkeys([bounce, *output]))
```

Be careful with integer entries in `output`: under the current `ricochet()` code, an integer is treated as energy, **not** as an emitted hole value.

Example:

```python
alphabet = tuple("abcdefghijklmnopqrstuvwxyz")
output = ("A", "B", "C")
bounce = 2
n = 20
cap = 10

model = ExactBallTrapModel(
    alphabet=alphabet,
    deflector_count=n,
    bounce=bounce,
    cap=cap,
    direction_range=10,
)

examples = make_examples([
    ("abc", ("A",)),
    ("cab", ("B",)),
])

analyzer = TrainabilityAnalyzer(
    model,
    examples,
    direction_min=-10,
    direction_max=10,
    value_domain=tuple(dict.fromkeys([bounce, *output])),
)

print(analyzer.parameter_space_log10())
print(analyzer.structural_checks())

# For small parameter spaces:
print(analyzer.report(analyzer.exact_enumeration()))

# For the real model, prefer:
# report = Z3TrainabilityAnalyzer(...).exact_smt()
```


In [ ]:
# Parameter-space demonstration for a modest model.
demo_model = ExactBallTrapModel(
    alphabet=tuple("ABC"),
    deflector_count=5,
    bounce=2,
    cap=10,
    direction_range=10,
)

demo_domain = tuple(dict.fromkeys([2, "X", "Y"]))

demo_analyzer = TrainabilityAnalyzer(
    demo_model,
    make_examples([("ABC", ("X",)), ("CBA", ("Y",))]),
    direction_min=-10,
    direction_max=10,
    value_domain=demo_domain,
)

space = demo_analyzer.model.parameter_space_size(
    demo_analyzer.direction_min,
    demo_analyzer.direction_max,
    demo_analyzer.value_domain,
)

print(f"Parameter assignments: {space:,}")
print(f"log10(parameter assignments): {math.log10(space):.2f}")
print("\nStructural analysis:")
for line in demo_analyzer.structural_checks():
    print(" ", line)


Parameter assignments: 879,071,484,375
log10(parameter assignments): 11.94

Structural analysis:


## 10. Comparing a trained brain to the exact model

You can take an already-trained `Ball_Trap_Brain` instance and convert it into a solver witness:

```python
def extract_params(brain):
    return BrainParams(
        tuple(brain.alphabet_directions),
        tuple(brain.alphabet_start_positions),
        tuple(d for d, v in brain.deflectors_list),
        tuple(v for d, v in brain.deflectors_list),
    )
```

Then:

```python
params = extract_params(trained_brain)
accuracy, predictions = analyzer.evaluate(params)
```

This is useful for the optimizer-vs-architecture question:

- If SMT says **SAT** and your trained brain is imperfect, optimization failed to reach a known-feasible solution.
- If SMT says **UNSAT**, no optimizer can produce a perfect solution inside those declared bounds.
- If SMT is not available and enumeration is too large, the honest result is **UNRESOLVED**.


In [ ]:
def extract_params(brain):
    return BrainParams(
        alphabet_directions=tuple(brain.alphabet_directions),
        alphabet_start_positions=tuple(brain.alphabet_start_positions),
        destinations=tuple(d for d, v in brain.deflectors_list),
        values=tuple(v for d, v in brain.deflectors_list),
    )

print("extract_params() is ready for an existing Ball_Trap_Brain instance.")


extract_params() is ready for an existing Ball_Trap_Brain instance.


# What this gives you

The analyzer gives you a formal workflow for your original idea:

\[
\boxed{\text{dataset} + \text{architecture} + \text{parameter bounds}}
\]

→ mathematical constraints

→

\[
\boxed{\text{feasible / infeasible / unresolved}}
\]

The most important capability is the distinction between:

\[
\text{no solution exists}
\]

and

\[
\text{a solution exists but my optimizer did not find it}.
\]

For the Ball Trap Brain, the finite `cap`, finite deflector count, bounded directions, bounded start positions, finite destination choices, and finite value choices make a bounded symbolic model possible.

The next research direction is to use the same encoding to study **expressive capacity** rather than only one dataset: enumerate or sample functions the architecture can realize, identify parameter symmetries, minimize the number of deflectors needed for a task, and search for architectural collisions/invariants that impose hard learning limits.
